# FHR-SAC on HumanoidStandup

MuJoCo `HumanoidStandup-v5` — stock SB3 SAC vs **FHRSAC** on the RL-Zoo `mujoco-defaults`
recipe (SB3 SAC defaults + `learning_starts 10000`, 1e6 steps, net 2×256).
The hardest of the four: 348-dim observations, 17-dim actions, no early termination, and a reward scale three to four orders of magnitude larger than the other envs (returns run into the 1e5 range). **That scale is the thing to watch** — the penalty is a Huber on the recurrence residual of Q, and Q here is O(1e5), so `penalty_raw` is enormous in absolute terms and the λ ladder should read very differently. Check §4's ρ table before drawing conclusions; expect the useful λ at or below the bottom rung.

The **baseline arm is bit-for-bit stock SB3 SAC** (`fhr_weight 0`, asserted by
`tests/test_sb3_sac_fhr.py`), so every baseline-vs-arm gap is attributable to
the recurrence penalty alone. HumanoidStandup is **not** in the RL-Zoo SAC hyperparameter file; the recipe is the zoo `mujoco-defaults` block used for Humanoid-v4, at 1e6 steps rather than Humanoid's 2e6, so there is no published reference score to match.

**Arm grid** — a direct λ × recurrence-order sweep, identical across all four
MuJoCo envs of this track:

| | r = 2 | r = 8 |
|---|---|---|
| **λ = 0.1** | exp1 | exp4 |
| **λ = 1**   | exp2 | exp5 |
| **λ = 10**  | exp3 | exp6 |

λ spans two decades so the sweep itself brackets the magnitude-matched point
(the value the fetch_reach probe pipeline solves for) instead of a probe run
having to find it — read `penalty_weighted` against `td_loss` in §4 to see
where that point actually landed. Order 2 is the floor (order-1 pure AR cannot
represent a Bellman-consistent sequence); order 8 tests whether a longer
recurrence window buys anything on a 1000-step episode.

**Update cadence is the zoo 1:1**, deliberately. The MountainCar tuning result
that transfers here is the *target rule* — "FHR is a stabiliser that licenses
aggressively fresh bootstrap targets" — and SAC ships that natively (Polyak
τ 0.005 every gradient step), so it costs nothing. The *burst-size* half of
that study lives in a separate ratio-matched family,
`configs/config_sb3_sac_burst.yaml`.

## 0 · Launch — train whatever the config defines

In [1]:
import pathlib, sys
SRC_RUNNERS = pathlib.Path.cwd().parent / "src"
if str(SRC_RUNNERS) not in sys.path:
    sys.path.insert(0, str(SRC_RUNNERS))
import run_sb3_seeds as runner
import yaml

CONFIG = "configs/config_sb3_sac.yaml"
# Every arm below — launched and analysed — is exactly what the config's
# experiment.fhr_experiments block currently defines; nothing is hardcoded.
EXPERIMENTS = sorted(int(k) for k in
                     (yaml.safe_load(open(CONFIG))["experiment"]
                      .get("fhr_experiments") or {}))
print("config experiments:", EXPERIMENTS)

LAUNCH = True
FORCE_EXP = False
if LAUNCH:
    manifest = runner.launch_all(config=CONFIG, experiments=EXPERIMENTS,
                                 max_workers=60, force=FORCE_EXP)
    print(sorted(manifest["runs"]))
else:
    print("LAUNCH = False — analysing existing runs only")

config experiments: [1, 2, 3, 4, 5, 6]
launching 14 run(s), 14 at a time: [('baseline', 44), ('baseline', 66), ('exp1', 44), ('exp1', 66), ('exp2', 44), ('exp2', 66), ('exp3', 44), ('exp3', 66), ('exp4', 44), ('exp4', 66), ('exp5', 44), ('exp5', 66), ('exp6', 44), ('exp6', 66)]


: 

In [ ]:
import csv, json, os
os.environ.setdefault("MUJOCO_GL", "egl")   # headless MuJoCo rendering (§7 videos)
import numpy as np
import matplotlib.pyplot as plt
import torch

REPO = pathlib.Path.cwd().parents[2]
sys.path.insert(0, str(REPO / "src"))

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "legend.frameon": False,
})
CFG = yaml.safe_load(open(CONFIG))
CMAN = json.load(open("cached/sb3_runs_manifest_sac.json"))
SEEDS = [str(s) for s in (CFG["experiment"].get("seeds")
                          or [CFG["experiment"]["seed"]])]

# ---- arms are DERIVED from the config, never hardcoded -------------------
# This family sweeps TWO dimensions (lambda and recurrence order), so unlike
# the fetch_reach notebook the arm key is (order, lambda): colour encodes
# lambda, linestyle encodes the order, and the two orders get their own
# comparison panel against the shared baseline.
LAM_COLOUR = {0.1: "tab:blue", 1.0: "tab:orange", 10.0: "tab:red"}
ORDER_LS = {2: "-", 8: "--"}
DEFAULTS = CFG["agent"]

def _spec(n, ov):
    lam = float(ov.get("fhr_weight", DEFAULTS["fhr_weight"]))
    order = int(ov.get("fhr_order", DEFAULTS["fhr_order"]))
    probe = float(ov.get("warmup_grad_steps", 0)) >= 1e8
    kind = "probe" if probe else "global"
    label = ("probe (lambda_eff = 0)" if probe
             else f"FHR  λ{lam:g} · r{order}")
    return {"arm": f"exp{n}", "kind": kind, "lam": lam, "order": order,
            "label": label}

specs = [{"arm": "baseline", "kind": "baseline", "lam": 0.0, "order": 0,
          "label": "baseline (stock SB3 SAC)"}]
unrun = []
for n, ov in sorted((int(k), v) for k, v in
                    (CFG["experiment"].get("fhr_experiments") or {}).items()):
    s = _spec(n, ov)
    (specs if CMAN["runs"].get(s["arm"]) else unrun).append(s)
specs.sort(key=lambda s: ({"baseline": 0, "probe": 1, "global": 2}[s["kind"]],
                          s["order"], s["lam"]))
if unrun:
    print("defined but not yet run:", [s["arm"] for s in unrun])

ARMS, INFO = {}, {}
for s in specs:
    label = s["label"]
    if label in ARMS:
        label = f"{label} [{s['arm']}]"
    s["colour"] = ("black" if s["kind"] == "baseline"
                   else LAM_COLOUR.get(s["lam"], "tab:purple"))
    s["ls"] = ORDER_LS.get(s["order"], "-")
    ARMS[label] = (s["arm"], s["colour"])
    INFO[label] = s

def labels_of(*kinds):
    return [l for l in ARMS if INFO[l]["kind"] in kinds]

def labels_order(order):
    return [l for l in ARMS if INFO[l]["kind"] == "global"
            and INFO[l]["order"] == order]

BASE = "baseline (stock SB3 SAC)" if "baseline (stock SB3 SAC)" in ARMS else None
GLOBALS_ = labels_of("global")
ORDERS = sorted({INFO[l]["order"] for l in GLOBALS_})

def run_dirs(arm):
    out = []
    for s in SEEDS:
        rel = CMAN["runs"].get(arm, {}).get(s)
        if rel and (pathlib.Path(rel) / "rewards.csv").exists():
            out.append((s, pathlib.Path(rel)))
    return out

def curves(arm, fname="eval.csv", x="env_steps", y="mean_reward"):
    cs = []
    for s, d in run_dirs(arm):
        if not (d / fname).exists():
            continue
        rows = list(csv.DictReader(open(d / fname)))
        if rows:
            cs.append((s, np.array([float(r[x]) for r in rows]),
                       np.array([float(r[y]) for r in rows])))
    return cs

def diag(arm, col):
    return curves(arm, fname="train_diagnostics.csv", x="episode", y=col)

print({l: ARMS[l][0] for l in ARMS})
# ---- shared sample-efficiency thresholds ---------------------------------
# Round reward levels (multiples of a 1/2/2.5/5 x 10^k unit, laddered up to
# the best arm's final seed-mean) — drawn as dotted horizontal lines on every
# learning-curve panel and quoted by the steps-to-threshold table in §2, so
# "reaches 150k at N steps" style claims read straight off either.
import math

def first_cross(x, y, thr):
    idx = np.argmax(y >= thr) if (y >= thr).any() else None
    return float(x[idx]) if idx is not None else np.nan

finals = {}
for label in ARMS:
    cs = curves(ARMS[label][0])
    if cs:
        finals[label] = np.array([y[-1] for _, _, y in cs])

THRESHOLDS = []
if finals:
    top = max(np.mean(v) for v in finals.values())
    mag = 10.0 ** math.floor(math.log10(top / 4))
    UNIT = min((m * mag for m in (1, 2, 2.5, 5, 10)),
               key=lambda c: abs(c - top / 4))
    THRESHOLDS = [UNIT * k for k in range(1, int(top // UNIT) + 1)]

def _fmt_thr(t):
    return f"{t/1000:g}k" if t >= 1000 else f"{t:g}"

def thr_lines(ax):
    for t in THRESHOLDS:
        ax.axhline(t, color="0.55", ls=":", lw=0.9, zorder=0)
        ax.annotate(_fmt_thr(t), xy=(0.997, t),
                    xycoords=("axes fraction", "data"), fontsize=7.5,
                    color="0.35", va="bottom", ha="right")

print("reward thresholds:", [_fmt_thr(t) for t in THRESHOLDS])


## 1 · Learning curves — one panel per recurrence order

Greedy evaluation (`eval.csv`): the deterministic policy on fixed reset seeds,
so the curves are paired across arms and the training stream is untouched.
Line = seed mean, band = seed min–max.

In [ ]:
def seed_band(ax, label, colr=None, ls=None, lw=1.9, alpha=0.95,
              band_alpha=0.16, legend=True, zorder=2, **kw):
    cs = curves(ARMS[label][0], **kw)
    if not cs:
        return
    n = min(len(y) for _, _, y in cs)
    x, Y = cs[0][1][:n], np.stack([y[:n] for _, _, y in cs])
    colr = colr or ARMS[label][1]
    ls = ls or INFO[label]["ls"]
    if len(cs) > 1:
        ax.fill_between(x, Y.min(0), Y.max(0), color=colr, alpha=band_alpha,
                        lw=0, zorder=zorder - 1)
    ax.plot(x, Y.mean(0), lw=lw, ls=ls, color=colr, alpha=alpha, zorder=zorder,
            label=f"{label} ({len(cs)} seeds)" if legend else None)

GROUPS = [(f"recurrence order r = {o}", labels_order(o)) for o in ORDERS]
GROUPS = [(t, ls) for t, ls in GROUPS if ls]
fig, axes = plt.subplots(1, len(GROUPS), figsize=(5.4 * len(GROUPS), 4.6),
                         sharey=True, squeeze=False)
for ax, (title, labels) in zip(axes.ravel(), GROUPS):
    if BASE:
        seed_band(ax, BASE, "gray", ls="-", lw=2.6, alpha=1.0, band_alpha=0.20,
                  zorder=1)
    for label in labels:
        seed_band(ax, label)
    ax.set(title=title, xlabel="environment steps")
    thr_lines(ax)
    ax.legend(fontsize=8, loc="upper left")
axes[0, 0].set_ylabel("greedy evaluation return")
fig.suptitle(f"{CFG['environment']['name']} greedy-eval curves — seed-mean, "
             f"band = seed min-max (seeds {', '.join(SEEDS)})",
             fontweight="bold", y=1.02)
plt.tight_layout()

### 1b · Each FHR arm vs the baseline — one panel per variant

The per-order panels above overlay three λ values; here every FHR arm gets its
own panel against the shared baseline, with the round reward thresholds drawn
as dotted horizontal lines and a dot where each seed-mean curve first crosses
them. A left-shifted dot on the same threshold line **is** the
sample-efficiency claim ("reaches X return in N steps, M steps before the
baseline") — exact numbers in §2's table.

In [ ]:
n = len(GLOBALS_)
ncols = min(3, max(1, n)); nrows = -(-n // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5.3 * ncols, 4.3 * nrows),
                         sharex=True, sharey=True, squeeze=False)
for ax in axes.ravel()[n:]:
    ax.set_visible(False)
for ax, label in zip(axes.ravel(), GLOBALS_):
    if BASE:
        seed_band(ax, BASE, "gray", ls="-", lw=2.4, alpha=1.0,
                  band_alpha=0.20, zorder=1)
    seed_band(ax, label, lw=2.3)
    thr_lines(ax)
    # dot = first crossing of the SEED-MEAN curve (matches the plotted line;
    # §2's table averages per-seed crossings instead, so its numbers can
    # differ slightly when seeds cross far apart)
    for lab in ([BASE] if BASE else []) + [label]:
        cs = curves(ARMS[lab][0])
        if not cs:
            continue
        m = min(len(y) for _, _, y in cs)
        x, ym = cs[0][1][:m], np.stack([y[:m] for _, _, y in cs]).mean(0)
        colr = "gray" if lab == BASE else ARMS[lab][1]
        for t in THRESHOLDS:
            xc = first_cross(x, ym, t)
            if np.isfinite(xc):
                ax.plot(xc, t, "o", ms=6.5, color=colr, mec="white", mew=0.9,
                        zorder=6)
    ax.set(title=label, xlabel="environment steps")
    ax.legend(fontsize=8, loc="lower right")
for ax in axes[:, 0]:
    ax.set_ylabel("greedy evaluation return")
fig.suptitle(f"{CFG['environment']['name']} — each FHR arm vs baseline "
             "(dots = seed-mean curve first crossing each threshold)",
             fontweight="bold", y=1.02)
plt.tight_layout()

## 2 · Sample efficiency — the claim FHR actually makes

Every prior FHR result in this repo is about **onset / sample efficiency**, not
asymptote. The table below reports, per arm, the environment steps needed to
first reach a set of return thresholds (seed-mean of the per-seed crossing;
`—` = never reached by that seed), plus the final greedy-eval return.

In [ ]:
# steps to first reach each threshold, per arm — the same round thresholds as
# the dotted lines above; per-seed crossings averaged (— when no seed gets
# there). finals / first_cross / THRESHOLDS come from the §0 loader cell.
if finals:
    hdr = "  ".join(f"{_fmt_thr(t):>9s}" for t in THRESHOLDS)
    print(f"{'arm':30s} {'final (seed mean)':>19s}   steps to reach: {hdr}")
    print("-" * (52 + 11 * len(THRESHOLDS)))
    for label in ARMS:
        cs = curves(ARMS[label][0])
        if not cs:
            continue
        fin = finals[label]
        cells_ = []
        for thr in THRESHOLDS:
            xs = [first_cross(x, y, thr) for _, x, y in cs]
            cells_.append("     —   " if np.all(np.isnan(xs))
                          else f"{np.nanmean(xs)/1000:8.0f}k")
        print(f"{label:30s} {fin.mean():10.1f} "
              f"[{fin.min():7.0f},{fin.max():7.0f}]   {'  '.join(cells_)}")

## 3 · Final performance and the λ × r grid

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4),
                         gridspec_kw={"width_ratios": [1.55, 1]})
ax = axes[0]
rows = [(l, finals[l]) for l in ARMS if l in finals]
ypos = np.arange(len(rows))[::-1]
for y, (label, v) in zip(ypos, rows):
    ax.barh(y, v.mean(), color=ARMS[label][1], alpha=0.85, height=0.62,
            hatch="//" if INFO[label]["order"] == 8 else None)
    ax.plot([v.min(), v.max()], [y, y], color="black", lw=1.4)
    ax.text(v.mean(), y, f"  {v.mean():.0f}", va="center", fontsize=9)
ax.set_yticks(ypos, [r[0] for r in rows], fontsize=9)
ax.set(title="final greedy-eval return (bar = seed mean, whisker = min-max)",
       xlabel="return")

# lambda x order grid of final return, as a fraction of the baseline
ax = axes[1]
lams = sorted({INFO[l]["lam"] for l in GLOBALS_})
grid = np.full((len(lams), len(ORDERS)), np.nan)
for l in GLOBALS_:
    if l in finals:
        grid[lams.index(INFO[l]["lam"]), ORDERS.index(INFO[l]["order"])] = \
            finals[l].mean()
base_mean = finals[BASE].mean() if BASE in finals else np.nan
rel = grid / base_mean if np.isfinite(base_mean) and base_mean != 0 else grid
im = ax.imshow(rel, cmap="RdYlGn", vmin=np.nanmin([0.5, np.nanmin(rel)]),
               vmax=np.nanmax([1.5, np.nanmax(rel)]), aspect="auto")
ax.set_xticks(range(len(ORDERS)), [f"r = {o}" for o in ORDERS])
ax.set_yticks(range(len(lams)), [f"λ = {l:g}" for l in lams])
for i in range(len(lams)):
    for j in range(len(ORDERS)):
        if np.isfinite(grid[i, j]):
            ax.text(j, i, f"{grid[i, j]:.0f}\n({rel[i, j]:.2f}×)",
                    ha="center", va="center", fontsize=9)
ax.set(title=f"final return vs baseline ({base_mean:.0f})")
ax.grid(False)
plt.colorbar(im, ax=ax, label="× baseline")
plt.tight_layout()

## 4 · FHR + SAC internals

The panel that decides whether the λ ladder was placed correctly is
**penalty_weighted vs td_loss**: λ·penalty is the fraction of the critic
objective the recurrence term actually owns. fetch_reach's calibrated pipeline
targets ρ = λ·penalty / td ∈ [0.5, 2]; here you can read off which rung of the
ladder landed in that band.

In [ ]:
panels = [("penalty_raw", "recurrence penalty (raw)", GLOBALS_, "log"),
          ("penalty_weighted", "λ · penalty (weighted)", GLOBALS_, "log"),
          ("td_loss", "critic TD loss", list(ARMS), "log"),
          ("sum_c", "sum of c", GLOBALS_, None),
          ("companion_radius", "companion spectral radius", GLOBALS_, None),
          ("residual_rms", "recurrence residual RMS", GLOBALS_, "log"),
          ("ent_coef", "temperature alpha", list(ARMS), None),
          ("actor_loss", "actor loss", list(ARMS), None),
          ("nan_skips", "nan skips", GLOBALS_, None)]
panels = [(c, t, ls, sc) for c, t, ls, sc in panels if ls]
fig, axes = plt.subplots(3, 3, figsize=(15.5, 11))
for ax, (col, title, labels, scale) in zip(axes.ravel(), panels):
    for label in labels:
        for s, x, y in diag(ARMS[label][0], col):
            m = np.isfinite(y)
            if m.any():
                ax.plot(x[m], y[m], lw=1.2, color=ARMS[label][1],
                        ls=INFO[label]["ls"],
                        alpha=0.85 if s == SEEDS[0] else 0.45,
                        label=label if s == SEEDS[0] else None)
    ax.set(title=title, xlabel="episode")
    if scale:
        ax.set_yscale(scale)
    ax.legend(fontsize=6.5)
for ax in axes.ravel()[len(panels):]:
    ax.axis("off")
plt.tight_layout()

In [ ]:
# rho = lambda * penalty / td_loss over the training tail, per arm — the
# number the fetch_reach calibration pipeline solves for directly.
print(f"{'arm':30s} {'median td':>12s} {'median λ·pen':>14s} {'ρ = λ·pen/td':>14s}")
print("-" * 74)
for label in [BASE] + GLOBALS_:
    if label is None:
        continue
    tds, pens = [], []
    for s, x, y in diag(ARMS[label][0], "td_loss"):
        tds.append(np.nanmedian(y[len(y) // 2:]))
    for s, x, y in diag(ARMS[label][0], "penalty_weighted"):
        pens.append(np.nanmedian(y[len(y) // 2:]))
    td = np.mean(tds) if tds else np.nan
    pen = np.mean(pens) if pens else np.nan
    rho = pen / td if td else np.nan
    print(f"{label:30s} {td:12.4g} {pen:14.4g} {rho:14.3g}")

## 5 · Rollout Hankel rank — the critic trace and the policy itself

Stacked per-rollout Hankels of the min-twin critic trace `Q(s_t, π(s_t))` and
of each action dimension of `π(s_t)`. A rank-r Hankel sequence satisfies an
order-r recurrence, so the measured rank of a *converged* policy is the
smallest order the penalty can enforce without fighting the solution — this is
what says whether r = 2 or r = 8 was the better-matched choice.

In [ ]:
from analysis.low_rank.continuous_rollout import hankel_rollout_continuous
from analysis.low_rank.rank import energy_rank

SHOW = [l for l in ([BASE] + GLOBALS_) if l]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
table = []
for label in SHOW:
    dirs = run_dirs(ARMS[label][0])
    if not dirs:
        continue
    _, adapter = runner.load_run_model(dirs[0][1], device="cpu")
    env = runner._make_env(CFG)
    mats = hankel_rollout_continuous(adapter, env, n_rollouts=3, base_seed=52)
    env.close()
    h_q, h_acts = mats[0], mats[1:]
    sv = np.linalg.svd(h_q, compute_uv=False); sv = sv / sv[0]
    rq = energy_rank(sv, 0.999)
    axes[0].semilogy(np.arange(1, min(len(sv), 30) + 1), sv[:30], lw=1.8,
                     color=ARMS[label][1], ls=INFO[label]["ls"],
                     label=f"{label} (rank {rq})")
    pi_ranks = []
    for j, h_a in enumerate(h_acts):
        sva = np.linalg.svd(h_a, compute_uv=False); sva = sva / sva[0]
        pi_ranks.append(energy_rank(sva, 0.999))
        if j == 0:
            axes[1].semilogy(np.arange(1, min(len(sva), 30) + 1), sva[:30],
                             lw=1.8, color=ARMS[label][1],
                             ls=INFO[label]["ls"],
                             label=f"{label} (rank {pi_ranks[0]})")
    table.append((label, rq, pi_ranks))
axes[0].set(title="Hankel(Q(s_t, π(s_t))) — critic trace", xlabel="index",
            ylabel="σ_i / σ_1")
axes[1].set(title="Hankel(π(s_t)[0]) — first action dim", xlabel="index")
for ax in axes:
    ax.legend(fontsize=7)
plt.tight_layout()
print(f"{'arm':30s} rank(Q)  ranks(π dims)")
for label, rq, pr in table:
    print(f"{label:30s} {rq:7d}  {pr}")

## 6 · GB10 cost

In [ ]:
import os
rows = []
for label in ARMS:
    for s, d in run_dirs(ARMS[label][0]):
        ck = d / "checkpoints" / "final.pt"
        if ck.exists():
            rows.append((label, s,
                         (os.path.getmtime(ck) - os.path.getmtime(d / "config.yaml")) / 60))
for label, s, mins in rows:
    print(f"{label:30s} seed {s}: {mins:6.1f} min")
base = [m for l, _, m in rows if l == BASE]
fhr = [m for l, _, m in rows if l != BASE]
if base and fhr:
    print(f"\nbaseline mean {np.mean(base):.1f} min; FHR-arm mean "
          f"{np.mean(fhr):.1f} min (overhead ×{np.mean(fhr)/np.mean(base):.2f})")

## 7 · Final policy rollouts — videos

How the final rollout is made: `record_final_videos` reloads each run's
**final checkpoint** (`checkpoints/final.pt` — the policy exactly as it stood
at the end of the training budget; the SB3 zip is self-describing, FHR
coefficients included), rebuilds the same `_make_env` wrapper stack the run
trained and eval'd on with `render_mode="rgb_array"`, and rolls **one greedy
episode** — the deterministic SAC actor (tanh-squashed mean, no sampling),
i.e. the same policy the `eval.csv` curves score — through gymnasium's
`RecordVideo`, which writes `<run_dir>/videos/epfinal-episode-0.mp4`. The
episode resets with the run's own seed, so each video is one representative
rollout per (arm, seed), not the 10-episode eval average. Only arms with
completed manifest runs appear; re-run after a sweep finishes to add the rest.

In [ ]:
from IPython.display import Video, display
from run_sb3_seeds import record_final_videos

for label in ([BASE] if BASE else []) + GLOBALS_:
    arm = ARMS[label][0]
    for seed, path in record_final_videos(arm, config=CONFIG):
        print(f"{label} ({arm}) — seed {seed}: {path}")
        display(Video(str(path), embed=True, width=420))

## 8 · Findings

*(fill in after the sweep completes)*

## Penalised-window Hankel rank (in-training probe)

Rank measured **where the penalty is applied** — on sampled replay windows
(anchor + `window_rank_lags` same-episode predecessors, online critic(s),
buffer actions) rather than on greedy on-policy rollouts. The probe runs for
every arm **including the λ=0 baseline**, which samples and measures the same
windows, so its curve is the control: if FHR operates as a rank constraint,
its arms should push the window rank / the penalty-block tail ratio *below*
the baseline on exactly these windows. Only instrumented runs
(`agent.window_rank_every > 0`, e.g. the `*_wrank` config family) carry
`window_hankel.csv`; older runs are skipped. Raw window matrices are in each
run's `window_matrices/` for offline re-analysis.


In [ ]:
# Penalised-window Hankel rank: scans every manifest under cached/ for runs
# carrying window_hankel.csv (the in-training probe's output) and, per
# family, plots rank + penalty-block tail ratio vs env steps for every arm.
import pathlib, sys
_repo = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "src" / "analysis").is_dir())
if str(_repo / "src") not in sys.path:
    sys.path.insert(0, str(_repo / "src"))
import numpy as np
import matplotlib.pyplot as plt
from analysis.low_rank.window_rank import (arm_tick_metrics,
                                           discover_probe_runs,
                                           final_quarter_summary)

_families = discover_probe_runs("cached")
if not _families:
    print("no instrumented runs under cached/ — launch a family with "
          "agent.window_rank_every > 0 (see the *_wrank config) first")
_palette = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple",
            "tab:cyan", "peru", "orchid", "tab:olive", "crimson"]
for _mname, _arms in _families.items():
    _fig, (_ax_r, _ax_p) = plt.subplots(1, 2, figsize=(13, 4.4))
    print(f"===== {_mname} =====")
    print(f"{'arm':24s} {'rank@99.9%':>10} {'rank@99%':>9} {'s2/s1':>8} "
          f"{'pen tail':>9}   (final-quarter means, seeds x critics)")
    _ci = 0
    for _arm, _dirs in _arms.items():
        _mets = arm_tick_metrics(_dirs)
        if not _mets:
            continue
        if _arm == "baseline":
            _col = "black"
        else:
            _col = _palette[_ci % len(_palette)]
            _ci += 1
        for _k, (_seed, _m) in enumerate(_mets):
            _lab = _arm if _k == 0 else None
            _lw = 2.0 if _arm == "baseline" else 1.2
            _ax_r.plot(_m["env_steps"], _m["rank999"], color=_col,
                       alpha=0.85, lw=_lw, label=_lab)
            _ax_p.semilogy(_m["env_steps"], _m["pen_tail_ratio"], color=_col,
                           alpha=0.85, lw=_lw, label=_lab)
        _s = final_quarter_summary(_mets)
        if _s:
            print(f"{_arm:24s} {_s['rank999']:10.1f} {_s['rank99']:9.1f} "
                  f"{_s['s2_s1']:8.4f} {_s['pen_tail_ratio']:9.4f}")
    _ax_r.set(title="energy rank @ 99.9% — penalised replay windows",
              xlabel="env steps", ylabel="rank")
    _ax_p.set(title="penalty-block tail  sigma_{r+1} / sigma_1",
              xlabel="env steps")
    for _ax in (_ax_r, _ax_p):
        _ax.legend(fontsize=7.5)
    _fig.suptitle(_mname, fontsize=10)
    plt.tight_layout()
    plt.show()
